# 04b · Entrenamiento GAN: StyleGAN2-ADA con entrenamiento transicional (Shahbazi et al., 2022), 128×128, reanudable

**Fase CRISP-DM:** 4. Modelado.

**Por qué este notebook reemplaza al 04:** con StyleGAN2-ADA condicional estándar, a los 500 kimg cada clase producía casi la misma imagen
(colapso de modos dentro de la clase). Es el fallo documentado por **Shahbazi et al. (2022), *Collapse by Conditioning: Training Class-conditional GANs with Limited Data*** (ICLR 2022):
con pocos datos por clase, el condicionamiento causa colapso, mientras que el entrenamiento no condicional es estable. Su solución:
**empezar no condicional e inyectar la clase gradualmente** en el generador y en la función objetivo.

**Técnica:** código oficial `mshahbazi72/transitional-cGAN` (basado en StyleGAN2-ADA PyTorch).
- λ(t) = 0 hasta `T_START` kimg (no condicional), sube linealmente hasta 1 en `T_END` kimg y luego se mantiene (condicional completo).
- Generador: la clase entra como λ·E(c) sumado tras la primera capa de la red de mapeo. Discriminador con rama no condicional + rama condicional (λ·pérdida condicional).
- **Escala del calendario:** los autores usan Ts = 2000 y Te = 4000 kimg, con Te ≈ mitad del entrenamiento. Con nuestro presupuesto se mantiene la **misma proporción**: total 2400 kimg, Ts = 600 (25 %), Te = 1200 (50 %). Se declara como adaptación.

**Igual que el 04:** `--cond=1 --mirror=0 --metrics=none --cfg=auto`, semilla 42, solo **train**, snapshots cada 100 kimg en Drive, reanudable.
**Parches:** los 5 del notebook 04 + **parche 6**: al reanudar, el código original sumaría de golpe todo el avance de λ desde 0 kimg; se corrige para que λ continúe desde el valor guardado.
W&B (registro en la nube que usa el repo) se desactiva.

**Duración:** 2400 kimg ≈ 17.6 h en L4 ≈ 32 unidades. GPU **L4**, versión **2026.07**.

## Paso 1 · Entorno

In [ ]:
import sys, os, subprocess, json, shutil, hashlib, re, time, glob
from pathlib import Path
from datetime import datetime, timezone
import torch
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print("Python:", sys.version.split()[0], "| PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "⚠️ SIN GPU")

Python: 3.12.13 | PyTorch: 2.11.0+cu128 | CUDA: 12.8
GPU: NVIDIA L4


## Paso 2 · Parámetros

In [ ]:
SEED = 42
TOTAL_KIMG = int(os.environ.get("TOTAL_KIMG", 2400))   # tope de entrenamiento
T_START = 600                                          # fin de la fase no condicional (25 %)
T_END = 1200                                           # condicionamiento completo (50 %)
SNAP_KIMG = int(os.environ.get("SNAP_KIMG", 100))      # cada cuánto se guarda un snapshot (≈ 43 min)
KIMG_PER_TICK = 4                                      # valor por defecto del repo
SNAP_TICKS = SNAP_KIMG // KIMG_PER_TICK
RUN_NAME = "transitional_cgan_cond128_seed42"

## Paso 3 · Drive, dataset y carpeta del entrenamiento

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/tesis_retail")
    LOCAL_DIR = Path("/content/data")
else:
    PROJECT_DIR = Path(os.environ.get("PROJECT_DIR", "./tesis_retail")).resolve()
    LOCAL_DIR = Path(os.environ.get("LOCAL_DIR", "./data_local")).resolve()
LOGS = PROJECT_DIR / "logs"
RUN_DIR = PROJECT_DIR / "checkpoints" / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

LOG03 = json.loads((LOGS / "03_piloto.json").read_text())
SG_ZIP = LOCAL_DIR / LOG03["datasets"]["train_zip"]
if not SG_ZIP.exists():
    shutil.copy2(PROJECT_DIR / "datasets" / SG_ZIP.name, SG_ZIP)
assert sha256_file(SG_ZIP) == LOG03["datasets"]["train_sha256"], "⚠️ El dataset no coincide con el del notebook 03"
print("Dataset verificado ✔ |", "carpeta del entrenamiento:", RUN_DIR)

Mounted at /content/drive
Dataset verificado ✔ | carpeta del entrenamiento: /content/drive/MyDrive/tesis_retail/checkpoints/transitional_cgan_cond128_seed42


## Paso 4 · Repo oficial transitional-cGAN + 6 parches

In [ ]:
%pip install -q click requests tqdm ninja psutil

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 24.2 MB/s eta 0:00:00


In [ ]:
SG_COMMIT = "b8792265ca23c6efd5e1c85c87cf85073c79266d"      # mshahbazi72/transitional-cGAN
SG3_COMMIT = "c233a919a6faee6e36a316ddd4eddababad1adf9"
SG_DIR = LOCAL_DIR / "transitional-cGAN"
SG3_DIR = LOCAL_DIR / "stylegan3"

def clonar(url, destino, commit):
    if not destino.exists():
        subprocess.run(["git", "clone", "-q", url, str(destino)], check=True)
    subprocess.run(["git", "-C", str(destino), "checkout", "-q", commit], check=True)
    subprocess.run(["git", "-C", str(destino), "checkout", "-q", "--", "."], check=True)   # archivos limpios
    head = subprocess.run(["git", "-C", str(destino), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    assert head == commit, head

clonar("https://github.com/mshahbazi72/transitional-cGAN.git", SG_DIR, SG_COMMIT)
clonar("https://github.com/NVlabs/stylegan3.git", SG3_DIR, SG3_COMMIT)

def parchar(ruta, viejo, nuevo, veces=1):
    s = ruta.read_text()
    assert s.count(viejo) == veces, f"Parche no aplicable en {ruta.name}: {viejo[:60]}"
    ruta.write_text(s.replace(viejo, nuevo))

# 1) distutils (Python >= 3.12)
parchar(SG_DIR / "dnnlib" / "util.py", "from distutils.util import strtobool",
        "try:\n    from distutils.util import strtobool\nexcept ImportError:  # Python >= 3.12\n"
        "    def strtobool(v):\n        return v.lower() in ('y', 'yes', 't', 'true', 'on', '1')")
# 2) Sampler.__init__ sin argumentos
parchar(SG_DIR / "torch_utils" / "misc.py", "        super().__init__(dataset)\n", "        super().__init__()\n")
# 3) grid_sample_gradfix de StyleGAN3 sin pkg_resources
s = (SG3_DIR / "torch_utils" / "ops" / "grid_sample_gradfix.py").read_text()
s = s.replace("from pkg_resources import parse_version\n", "")
s = s.replace("_use_pytorch_1_11_api = parse_version(torch.__version__) >= parse_version('1.11.0a') # Allow prerelease builds of 1.11",
              "_TV = tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2])\n_use_pytorch_1_11_api = _TV >= (1, 11)")
s = s.replace("_use_pytorch_1_12_api = parse_version(torch.__version__) >= parse_version('1.12.0a') # Allow prerelease builds of 1.12",
              "_use_pytorch_1_12_api = _TV >= (1, 12)")
assert "parse_version" not in s
(SG_DIR / "torch_utils" / "ops" / "grid_sample_gradfix.py").write_text(s)
# 4) custom_ops: usar el módulo que devuelve cpp_extension.load
co = SG_DIR / "torch_utils" / "custom_ops.py"
parchar(co, "            torch.utils.cpp_extension.load(name=module_name, build_directory=build_dir,",
            "            module = torch.utils.cpp_extension.load(name=module_name, build_directory=build_dir,")
parchar(co, "            torch.utils.cpp_extension.load(name=module_name, verbose=verbose_build, sources=sources, **build_kwargs)",
            "            module = torch.utils.cpp_extension.load(name=module_name, verbose=verbose_build, sources=sources, **build_kwargs)")
parchar(co, "        module = importlib.import_module(module_name)\n", "")
# 5) reanudación: restaurar contador de kimg y p de ADA; mantener ada_kimg=500
tl = SG_DIR / "training" / "training_loop.py"
parchar(tl, "        augment_pipe.p.copy_(torch.as_tensor(augment_p))\n",
            "        augment_pipe.p.copy_(torch.as_tensor(augment_p))\n"
            "        if (resume_pkl is not None) and (resume_data.get('augment_pipe') is not None):  # parche 5\n"
            "            augment_pipe.p.copy_(resume_data['augment_pipe'].p)\n"
            "            print(f'ADA p restaurado: {float(augment_pipe.p):.4f}')\n")
parchar(tl, "    cur_nimg = 0\n", "    cur_nimg = int(os.environ.get('SG_RESUME_KIMG', '0')) * 1000  # parche 5\n")
parchar(SG_DIR / "train.py", "        args.ada_kimg = 100 # make ADA react faster at the beginning",
        "        if not os.environ.get('SG_RESUME_KIMG'): args.ada_kimg = 100 # parche 5: solo en transfer learning")
# 6) transición: al reanudar, old_nimg debe partir del contador restaurado (si no, λ salta de golpe)
parchar(tl, "    old_nimg = 0    # Added by the authors\n", "    old_nimg = cur_nimg  # parche 6\n")
os.environ["WANDB_MODE"] = "disabled"
print("6 parches aplicados ✔")

os.environ["TORCH_EXTENSIONS_DIR"] = str(LOCAL_DIR / "torch_extensions")
if torch.cuda.is_available():
    cc = torch.cuda.get_device_capability(0)
    os.environ["TORCH_CUDA_ARCH_LIST"] = f"{cc[0]}.{cc[1]}"

6 parches aplicados ✔


## Paso 5 · ¿Empezar o reanudar?

In [ ]:
def snapshots():
    out = []
    for p in RUN_DIR.glob("*/network-snapshot-*.pkl"):
        m = re.search(r"network-snapshot-(\d+)\.pkl$", p.name)
        if m and p.stat().st_size > 0:
            out.append((int(m.group(1)), p))
    return sorted(out)

previos = snapshots()
if previos:
    RESUME_KIMG, RESUME_PKL = previos[-1]
    print(f"Se reanuda desde {RESUME_KIMG} kimg → {RESUME_PKL}")
else:
    RESUME_KIMG, RESUME_PKL = 0, None
    print("No hay snapshots previos: entrenamiento nuevo.")
if RESUME_KIMG >= TOTAL_KIMG:
    print(f"✔ Ya se alcanzó el tope ({RESUME_KIMG} ≥ {TOTAL_KIMG} kimg). No hay nada que entrenar.")

No hay snapshots previos: entrenamiento nuevo.


## Paso 6 · Entrenar
La celda muestra una línea de progreso cada ~5 min (kimg, s/kimg, `augment` = probabilidad de ADA). El log completo queda en Drive.
Puedes detenerla en cualquier momento (botón ■): el último snapshot guardado queda en Drive y se reanuda desde él.

In [ ]:
env = os.environ.copy()
cmd = [sys.executable, "train.py", f"--outdir={RUN_DIR}", f"--data={SG_ZIP}", "--gpus=1",
       "--cond=1", "--mirror=0", "--metrics=none", "--cfg=auto",
       f"--kimg={TOTAL_KIMG}", f"--snap={SNAP_TICKS}", f"--seed={SEED + RESUME_KIMG}",
       f"--t_start_kimg={T_START}", f"--t_end_kimg={T_END}"]
if RESUME_PKL is not None:
    cmd.append(f"--resume={RESUME_PKL}")
    env["SG_RESUME_KIMG"] = str(RESUME_KIMG)

LOG_PATH = LOGS / f"04b_gan_{datetime.now().strftime('%Y%m%d_%H%M')}_desde{RESUME_KIMG}kimg.log"
print(" ".join(cmd)); print("Log:", LOG_PATH)

if RESUME_KIMG < TOTAL_KIMG:
    t0 = time.time(); ultima = ""
    with open(LOG_PATH, "w") as flog:
        proc = subprocess.Popen(cmd, cwd=SG_DIR, stdout=flog, stderr=subprocess.STDOUT, text=True, env=env)
        try:
            while proc.poll() is None:
                time.sleep(300 if IN_COLAB else 5)
                ticks = re.findall(r"(tick\s+\d+\s+kimg\s+[\d.]+.*)", LOG_PATH.read_text(errors="ignore"))
                if ticks and ticks[-1] != ultima:
                    ultima = ticks[-1]
                    campos = re.search(r"kimg\s+([\d.]+).*?sec/kimg\s+([\d.]+).*?augment\s+([\d.]+)", ultima)
                    if campos:
                        k, spk, aug = campos.groups()
                        restante = (TOTAL_KIMG - float(k)) * float(spk) / 3600
                        lam = min(max((float(k) - T_START) / (T_END - T_START), 0.0), 1.0)
                        print(f"[{datetime.now().strftime('%H:%M')}] kimg {k} | {spk} s/kimg | ADA p {aug} | λ cond {lam:.2f} | faltan ≈ {restante:.1f} h", flush=True)
        except KeyboardInterrupt:
            proc.terminate(); proc.wait(timeout=60)
            print("Detenido manualmente. El último snapshot guardado está en Drive.")
    print(f"Proceso terminado (código {proc.returncode}) tras {(time.time()-t0)/3600:.2f} h")
    print("\n".join(LOG_PATH.read_text(errors="ignore").splitlines()[-8:]))

/usr/bin/python3 train.py --outdir=/content/drive/MyDrive/tesis_retail/checkpoints/transitional_cgan_cond128_seed42 --data=/content/data/train128_stylegan.zip --gpus=1 --cond=1 --mirror=0 --metrics=none --cfg=auto --kimg=2400 --snap=25 --seed=42 --t_start_kimg=600 --t_end_kimg=1200
Log: /content/drive/MyDrive/tesis_retail/logs/04b_gan_20261009_2209_desde0kimg.log
[22:14] kimg 0.0 | 2060.51 s/kimg | ADA p 0.000 | λ cond 0.00 | faltan ≈ 1373.7 h
[22:19] kimg 12.0 | 26.26 s/kimg | ADA p 0.015 | λ cond 0.00 | faltan ≈ 17.4 h
[22:24] kimg 24.0 | 26.20 s/kimg | ADA p 0.029 | λ cond 0.00 | faltan ≈ 17.3 h
[22:29] kimg 36.0 | 26.27 s/kimg | ADA p 0.045 | λ cond 0.00 | faltan ≈ 17.3 h
[22:34] kimg 48.0 | 26.40 s/kimg | ADA p 0.064 | λ cond 0.00 | faltan ≈ 17.2 h
[22:39] kimg 60.0 | 26.31 s/kimg | ADA p 0.079 | λ cond 0.00 | faltan ≈ 17.1 h
[22:44] kimg 72.0 | 26.35 s/kimg | ADA p 0.094 | λ cond 0.00 | faltan ≈ 17.0 h
[22:49] kimg 80.0 | 26.42 s/kimg | ADA p 0.106 | λ cond 0.00 | faltan ≈ 17.0 h

## Paso 7 · Resumen y registro

In [ ]:
snaps = snapshots()
print(f"Snapshots en Drive: {len(snaps)} | último: {snaps[-1][0] if snaps else 0} kimg")
for k, p in snaps[-5:]:
    print(f"  {k:>6} kimg  {p.parent.name}/{p.name}  ({p.stat().st_size/2**20:.0f} MB)")
grids = sorted(RUN_DIR.glob("*/fakes*.png"))
if grids:
    print("Última malla de muestras:", grids[-1])
    if IN_COLAB:
        from IPython.display import Image as IPImage, display
        display(IPImage(filename=str(grids[-1]), width=900))

registro = LOGS / "04b_gan_registro.json"
hist = json.loads(registro.read_text()) if registro.exists() else {"sesiones": []}
hist.update({"run": RUN_NAME, "total_kimg_objetivo": TOTAL_KIMG, "snap_kimg": SNAP_KIMG,
             "commits": {"transitional_cgan": SG_COMMIT, "stylegan3_gradfix": SG3_COMMIT},
             "parches": ["distutils", "Sampler", "grid_sample_gradfix (StyleGAN3)", "custom_ops", "reanudacion kimg/ADA p", "reanudacion transicion old_nimg"],
             "transicion_kimg": [T_START, T_END], "tecnica": "Transitional cGAN (Shahbazi et al., 2022) sobre StyleGAN2-ADA",
             "opciones": "--cond=1 --mirror=0 --metrics=none --cfg=auto"})
hist["sesiones"].append({"fin_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
                         "desde_kimg": RESUME_KIMG, "hasta_kimg": snaps[-1][0] if snaps else 0,
                         "log": LOG_PATH.name, "versiones": {"python": sys.version.split()[0], "torch": torch.__version__}})
registro.write_text(json.dumps(hist, indent=2, ensure_ascii=False))
print("Registro actualizado:", registro)

**Al terminar:** desconecta la GPU (*Entorno de ejecución → Desconectar y borrar*).

In [ ]:
from google.colab import runtime
runtime.unassign()